## Llama 3.1 8B zero-shot structured-output

This notebook uses Outlines and the repository's current prompt and Pydantic schema to extract structured valid JSON from the synthetic free-text rectal MRI reports.

In [ ]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'llm_benchmark_example':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from llm_benchmark_example.prompt import SYSTEM_PROMPT
from llm_benchmark_example.pydantic_template import RectalCancerReport

In [ ]:
DATA_PATH = PROJECT_ROOT / 'generated_datasets' / 'updated_synthetic_dataset.csv'
OUTPUT_DIR = PROJECT_ROOT / 'generated_datasets' / 'llm_benchmark_outputs'

df = pd.read_csv(DATA_PATH)
required_columns = {'structured_report', 'free_text_report'}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'Missing required dataset columns: {sorted(missing_columns)}')

print(f'Loaded {len(df):,} reports from {DATA_PATH}')

## Inference environment

The original inference was run on premises in a Docker container using an NVIDIA RTX A6000 GPU. A similar experiment can be conducted in Google Colab by selecting a GPU runtime.

CUDA diagnostics and model loading are intentionally commented out below. Update `MODEL_NAME_OR_PATH` for the target environment and uncomment the cell before running the benchmark.

In [ ]:
# import outlines
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# print(f'CUDA available: {torch.cuda.is_available()}')
# print(f'CUDA device count: {torch.cuda.device_count()}')
# if torch.cuda.is_available():
#     print(f'CUDA device: {torch.cuda.get_device_name(0)}')

# MODEL_NAME_OR_PATH = '/workspace/models_params/llama/Llama-3.1-8B-Instruct'
# # For Google Colab, use the corresponding Hugging Face model ID after
# # configuring model access and authentication.
# hf_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME_OR_PATH)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH)
# hf_model = hf_model.to('cuda:0')
# llm = outlines.from_transformers(hf_model, tokenizer)
# report_processor = outlines.Generator(llm, RectalCancerReport)

In [ ]:
SCHEMA = json.dumps(RectalCancerReport.model_json_schema(), indent=2)

def build_zero_shot_prompt():
    return SYSTEM_PROMPT.format(schema=SCHEMA)

def format_chat(system_prompt, report):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': f'REPORT: {report}\n\n### OUTPUT:'},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

In [ ]:
def parse_structured_result(result):
    if isinstance(result, RectalCancerReport):
        return result.model_dump()
    return RectalCancerReport.model_validate_json(result).model_dump()

def run_benchmark(input_df, system_prompt, output_column):
    if 'tokenizer' not in globals() or 'report_processor' not in globals():
        raise RuntimeError('Uncomment and run the CUDA/model-loading cell first.')

    outputs = []
    durations = []
    for position, report in enumerate(input_df['free_text_report'], start=1):
        prompt = format_chat(system_prompt, report)
        started_at = time.perf_counter()
        result = report_processor(prompt)
        elapsed = time.perf_counter() - started_at
        outputs.append(parse_structured_result(result))
        durations.append(elapsed)
        print(f'Processed report {position:,}/{len(input_df):,} in {elapsed:.1f} seconds')

    output_df = input_df.copy()
    output_df[output_column] = [
        json.dumps(item, ensure_ascii=False) for item in outputs
    ]
    mean_duration = sum(durations) / len(durations)
    print(f'Average inference time: {mean_duration:.1f} seconds per report')
    return output_df, durations

def save_results(output_df, filename):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / filename
    output_df.to_csv(output_path, index=False)
    print(f'Saved results to {output_path}')
    return output_path

## Zero-shot

In [ ]:
zero_shot_df, zero_shot_durations = run_benchmark(
    input_df=df,
    system_prompt=build_zero_shot_prompt(),
    output_column='llama3_1_8b_zero_shot_json',
)
zero_shot_path = save_results(
    zero_shot_df, 'llama3_1_8b_zero_shot.csv'
)